## Step 02 - First Research Agent

Goal: wrap the same prompt into `Agent + Task + Crew`.

### What's New in This Step

- Step 01 called the LLM directly with one prompt.
- This step adds an `Agent` (role + goal), a `Task` (work contract), and a `Crew` (runner).
- Even with this structure, output is still limited by LLM training knowledge because no external tool is connected yet.

In [ ]:
import os
from dotenv import load_dotenv
from crewai import LLM, Agent, Crew, Task

load_dotenv()

TOPIC = "Platform Engineering Best Practices"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_api_key:
    raise ValueError("Missing OPENROUTER_API_KEY")


In [ ]:
# LLM: the reasoning engine used by all agents in this notebook.
llm = LLM(
    model="openai/gpt-4o",
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)

# Agent: defines who performs the work and how it should behave.
research_agent = Agent(
    role="Research Analyst",
    goal="Find latest updates on {topic}",
    backstory="You give short and clear summaries.",
    llm=llm,
    verbose=False,
)


### Why role / goal / backstory matter

These three fields are not labels — they are concatenated into the system prompt the LLM sees on every call.

- `role` shapes the persona ("Research Analyst" vs. "Skeptical Reviewer" produces very different tone).
- `goal` is the agent's north star; `{topic}` here is templated from `crew.kickoff(inputs=...)`.
- `backstory` is the cheapest knob for steering style — try changing "short and clear summaries" to "long, narrative explanations" and rerun to see the effect.

In [ ]:
# Task: defines the exact assignment and output expectation for the agent.
research_task = Task(
    description="Research {topic} from 2026 onward in short bullet points.",
    expected_output="As short 8 to 10 bullets of research summary.",
    agent=research_agent,
)

# Crew: orchestrates execution order for one or more tasks.
crew = Crew(
    agents=[research_agent],
    tasks=[research_task],
    verbose=False,
)

result = crew.kickoff(inputs={"topic": TOPIC})
print(getattr(result, "raw", str(result)))


### Recap

- LLM did: generate output from its internal/trained knowledge.
- Agent did: frame the behavior (who it is and what it should do), but still relied on the LLM alone.
- Task enforced: scope and output shape; external freshness is added next in Step 03 via tools.

